In [1]:
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import os

In [3]:
INPUT_DIR = "../inputs/"

# List all files in the input directory
files = os.listdir(INPUT_DIR)

def get_mat_dims(input_file_name):
    # File name is of the form "input_matrix_{nrows:04d}_{ncols:04d}.bin"
    parts = input_file_name.replace(".bin", "").split("_")
    nrows, ncols = int(parts[2]), int(parts[3])
    return nrows, ncols

MAT_DIMS = [get_mat_dims(f) for f in files]

# Sort MAT_DIMS by the product of nrows and ncols
MAT_DIMS.sort(key=lambda x: x[0] * x[1])

print(MAT_DIMS)

[(76, 76), (13, 620), (69, 153), (585, 196), (168, 742), (391, 391), (455, 338), (137, 1163), (633, 633), (1264, 349), (592, 1618), (1221, 1150), (1308, 1308), (1514, 1514), (1655, 1594)]


In [4]:
def get_time_from_log(log_file_name):
    with open(log_file_name, "r") as f:
        lines = f.readlines()

    # Rstrip '\n' from each line
    lines = [line.rstrip() for line in lines]

    if lines[0] == "PASSED":
        tm_taken = float(lines[1].split()[-2]) / 1000.0
        return tm_taken
    return None

# Experiment 1
Fix number of threads, varying size of the input

In [ ]:
NUM_THRS = 8

tm_data = dict()

ALGOS = ["gk", "qr"]
for algo in ALGOS:
    tm_data[algo] = []

# Create a directory for logs if it doesn't exist
if not os.path.exists("exp1_logs"):
    os.makedirs("exp1_logs")

for i, (nrows, ncols) in enumerate(MAT_DIMS):
    for ALGO in ALGOS:
        log_file_name = f"exp1_logs/log_{nrows:04d}_{ncols:04d}_{ALGO}.txt"

        # Run QR algorithm
        subprocess.run(f"./{ALGO} {nrows} {ncols} {NUM_THRS} > {log_file_name}", shell=True)

        # Get the time taken from the log file
        tm_taken = get_time_from_log(log_file_name)

        if tm_taken is not None:
            tm_data[ALGO].append(tm_taken)
        else:
            print(f"Test failed for {nrows}x{ncols} matrix")
            tm_data[ALGO].append(-1.0)

# Plotting
plt.figure(figsize=(10, 6))

for ALGO in ALGOS:
    dat = tm_data[ALGO]
    # Discard the ones with -1 time
    dat = [d for d in dat if d != -1.0]
    # Get the corresponding matrix dimensions
    mat_dims = [MAT_DIMS[i] for i in range(len(MAT_DIMS)) if tm_data[ALGO][i] != -1.0]
    # Get the product of nrows and ncols
    mat_sizes = [nrows * ncols for nrows, ncols in mat_dims]
    # Plot the data
    plt.plot(mat_sizes, dat, label=ALGO)

    print(f"Algorithm: {ALGO}")
    print(f"mat_dims: {mat_dims}")
    print(f"tm_data: {dat}")
    
# Set x-axis to log scale
plt.xscale("log")
plt.yscale("log")
plt.xticks(mat_sizes, [f"{nrows}x{ncols}" for nrows, ncols in mat_dims], rotation=45)
# Set font size to 14
plt.rcParams.update({'font.size': 14})
plt.title("Time taken vs Matrix Size")
plt.xlabel("Matrix Size (nrows x ncols)")
plt.ylabel("Time taken (ms)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("exp1_plot.png")
plt.show()

# Experiment 2
Fix size of the input, varying number of threads

In [ ]:
NUM_THRS = [1, 2, 4, 8, 16, 32, 64, 128]
tm_data = []

nrows, ncols = 633, 633

tm_data = dict()

ALGOS = ["gk", "qr"]
for algo in ALGOS:
    tm_data[algo] = []

# Create a directory for logs if it doesn't exist
if not os.path.exists("exp2_logs"):
    os.makedirs("exp2_logs")


for num_thrs in NUM_THRS:
    for ALGO in ALGOS:
        log_file_name = f"exp2_logs/log_{nrows:04d}_{ncols:04d}_{num_thrs}_{ALGO}.txt"

        # Run QR algorithm
        subprocess.run(f"./{ALGO} {nrows} {ncols} {num_thrs} > {log_file_name}", shell=True)

        # Get the time taken from the log file
        tm_taken = get_time_from_log(log_file_name)

        if tm_taken is not None:
            tm_data[ALGO].append(tm_taken)
        else:
            print(f"Test failed for {nrows}x{ncols} matrix")
            tm_data[ALGO].append(-1.0)

# Plotting
plt.figure(figsize=(10, 6))
for ALGO in ALGOS:
    dat = tm_data[ALGO]
    # Discard the ones with -1 time
    dat = [d for d in dat if d != -1.0]
    # Get the corresponding matrix dimensions
    mat_dims = [NUM_THRS[i] for i in range(len(NUM_THRS)) if tm_data[ALGO][i] != -1.0]
    # Plot the data
    plt.plot(mat_dims, dat, label=ALGO)

    print(f"Algorithm: {ALGO}")
    print(f"tm_data: {dat}")
    print(f"num_thrs: {NUM_THRS}")
    print(f"mat_dims: {mat_dims}")

# Set x-axis to log scale
plt.xscale("log")
plt.yscale("log")
# Set font size to 14
plt.rcParams.update({'font.size': 14})
plt.title("Time taken vs Number of Threads")
plt.xlabel("Number of Threads")
plt.ylabel("Time taken (ms)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("exp2_plot.png")
plt.show()